# Generative AI: Assignment 1

**Total Marks:** 100 | **Due Date:** 14th September, 9PM

This notebook covers:
- **Part 1:** Topic Detection & Summarization of BBC News Articles (45 marks)
- **Part 2:** Job Postings Analysis - Role Categorization & Requirements Extraction (55 marks)

## Initial Setup
### Set API key for Groq
Click [here](https://console.groq.com/keys) to create an API key for Groq, if not already created.

In [4]:
import os, json, re, getpass
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

False

In [5]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [6]:
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0)

---
# Part 1: Topic Detection and Summarization of News Articles


## Step 1: Load the Dataset
Using the BBC News Full-Text dataset, limited to the first 30 articles.

In [7]:
news_df = pd.read_csv("bbc-news-data.csv", sep="\t")
news_df = news_df.head(30).reset_index(drop=True)
print(news_df.shape)
news_df.head()

(30, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Step 2: Define the Topic Classification Task 

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

topic_categories = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

classification_template = ChatPromptTemplate([
    ("system", "You are a news editor who classifies articles into a single topic category."),
    ("human", """Analyze the following news article and identify its topic as one of the following categories: {categories}.

Examples:
Article: "The prime minister announced new legislation in parliament today..." -> Politics
Article: "The striker scored a hat-trick to win the match for his team..." -> Sport

Article:
{article}

Return ONLY the single category label, nothing else."""),
])

classification_chain = classification_template | llm | StrOutputParser()

In [9]:
# Show this works for a sample datapoint
sample_article = news_df.loc[0, "content"]

sample_topic = classification_chain.invoke({
    "categories": ", ".join(topic_categories),
    "article": sample_article
}).strip()

print("Predicted Topic:", sample_topic)
print("Actual Category:", news_df.loc[0, "category"])

Predicted Topic: Business
Actual Category: business


## Step 3: Define the Summarization Task

In [10]:
summarization_template = ChatPromptTemplate([
    ("system", "You are a skilled news summarizer."),
    ("human", """Summarize the main points of the following news article in 2-3 sentences.
Capture the who/what/when/where/why as applicable, without adding personal commentary.

Article:
{article}"""),
])

summarization_chain = summarization_template | llm | StrOutputParser()

In [11]:
# Show this works for a sample datapoint
sample_summary = summarization_chain.invoke({"article": sample_article}).strip()
print(sample_summary)

TimeWarner reported a 76% jump in fourth‑quarter profit to $1.13 billion for the three months ended December, with sales rising 2% to $11.1 billion, driven by higher high‑speed internet connections, stronger advertising revenue and one‑off gains, while the company now holds an 8% stake in Google and its film division saw profits fall 27% after box‑office flops. The firm also disclosed a loss of 464,000 AOL subscribers despite an 8% rise in underlying AOL profit, announced a $300 million settlement with the SEC and will restate its 2000 and 2003 results as part of the regulator’s investigation, while projecting about 5% operating‑earnings growth for 2005.


## Step 4: Key Entity Extraction


In [12]:
from pydantic import BaseModel, Field
from typing import List

class KeyEntities(BaseModel):
    """Important entities mentioned in a news article."""
    people: List[str] = Field(description="Notable people mentioned in the article")
    organizations: List[str] = Field(description="Organizations/companies mentioned in the article")
    locations: List[str] = Field(description="Places/locations mentioned in the article")

entity_template = ChatPromptTemplate([
    ("system", "You extract key named entities from news articles."),
    ("human", """From the article below, list the names of any important people, organizations, or places mentioned.

Article:
{article}"""),
])

entity_chain = entity_template | llm.with_structured_output(KeyEntities)

In [13]:
# Show this works for a sample datapoint
sample_entities = entity_chain.invoke({"article": sample_article})
sample_entities

KeyEntities(people=['Richard Parsons'], organizations=['TimeWarner', 'Google', 'AOL', 'Warner Bros', 'U.S. Securities and Exchange Commission (SEC)', 'Bertelsmann', 'AOL Europe'], locations=['United States', 'Germany'])

## Step 5: Update the DataFrame with Results
Combine all three tasks into a single structured chain (fewer LLM calls, more efficient) and apply it across the first 30 articles.

In [14]:
class ArticleAnalysis(BaseModel):
    """Structured analysis of a news article: topic, summary and key entities."""
    Detected_Topic: str = Field(description=f"The single best-fit topic, one of: {', '.join(topic_categories)}")
    Summary: str = Field(description="A concise 2-3 sentence summary of the article")
    Key_Entities: List[str] = Field(description="Important people, organizations, or locations mentioned in the article")

analysis_template = ChatPromptTemplate([
    ("system", "You are an expert news analyst."),
    ("human", """Analyze the following news article and provide:
1. Its topic - one of: {categories}
2. A 2-3 sentence summary capturing the key points
3. A list of key entities (notable people, organizations, or locations)

Article:
{article}"""),
])

analysis_chain = analysis_template | llm.with_structured_output(ArticleAnalysis)

In [15]:
results = []
for idx, row in news_df.iterrows():
    try:
        analysis = analysis_chain.invoke({
            "categories": ", ".join(topic_categories),
            "article": row["content"]
        })
        results.append(analysis.model_dump())
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        results.append({"Detected_Topic": "Not specified", "Summary": "Not specified", "Key_Entities": []})

results_df = pd.DataFrame(results)
results_df.head()

,Detected_Topic,Summary,Key_Entities
0,Business,TimeWarner reported a 76% jump in quarterly pr...,"[TimeWarner, Google, AOL, Warner Bros, Richard..."
1,Business,The dollar rose to a three‑month high against ...,"[Federal Reserve, Alan Greenspan, Bank of Amer..."
2,Business,"Menatep Group, the owner of the former Yukos p...","[Yukos, Menatep Group, Rosneft, Yugansk, Mikha..."
3,Business,British Airways reported a 40% drop in pre‑tax...,"[British Airways, Rod Eddington, Martin Brough..."
4,Business,Shares in UK drinks and food company Allied Do...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."


In [16]:
# Final merged dataframe with all original and new columns together
news_final_df = pd.concat([news_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
news_final_df.head()

,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,TimeWarner reported a 76% jump in quarterly pr...,"[TimeWarner, Google, AOL, Warner Bros, Richard..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,The dollar rose to a three‑month high against ...,"[Federal Reserve, Alan Greenspan, Bank of Amer..."
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,"Menatep Group, the owner of the former Yukos p...","[Yukos, Menatep Group, Rosneft, Yugansk, Mikha..."
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways reported a 40% drop in pre‑tax...,"[British Airways, Rod Eddington, Martin Brough..."
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Shares in UK drinks and food company Allied Do...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."


In [17]:
news_final_df.to_csv("part1_news_analysis_results.csv", index=False)
news_final_df.shape

(30, 7)

---
# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction


## Step 1: Load the Dataset
Using the job postings dataset, limited to the first 25 postings.

In [18]:
jobs_df = pd.read_csv("job_title_des.csv")
jobs_df = jobs_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
jobs_df = jobs_df[["Job_Title", "Job_Description"]].head(25).reset_index(drop=True)
print(jobs_df.shape)
jobs_df.head()

(25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Define the Job Category Classification Task

In [19]:
job_categories = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Sales", "Operations", "Other"]

job_classification_template = ChatPromptTemplate([
    ("system", "You are an HR specialist who categorizes job postings into a broad domain."),
    ("human", """Given the following job title and description, categorize the job into one of the following domains: {categories}.
If unsure, use "Other".

Job: {job_title}
Description: {job_description}

Return ONLY the single domain category label, nothing else."""),
])

job_classification_chain = job_classification_template | llm | StrOutputParser()

In [20]:
# Show this works for a sample datapoint
sample_job_title = jobs_df.loc[0, "Job_Title"]
sample_job_desc = jobs_df.loc[0, "Job_Description"]

sample_category = job_classification_chain.invoke({
    "categories": ", ".join(job_categories),
    "job_title": sample_job_title,
    "job_description": sample_job_desc
}).strip()

print("Job Title:", sample_job_title)
print("Predicted Category:", sample_category)

Job Title: Flutter Developer
Predicted Category: Technology/IT


## Step 3: Define the Requirements Extraction Task

In [21]:
class JobRequirements(BaseModel):
    """Key requirements extracted from a job description."""
    Required_Skills: List[str] = Field(description="Key skills, programming languages, tools or domain knowledge mentioned. Empty list if none.")
    Education_Required: str = Field(description="Minimum education level required/preferred (e.g. Bachelor's, MBA). 'Not specified' if not mentioned.")
    Experience_Required: str = Field(description="Years of experience or experience level required (e.g. '3+ years'). 'Not specified' if not mentioned.")

requirements_template = ChatPromptTemplate([
    ("system", "You extract structured hiring requirements from job descriptions."),
    ("human", """Extract the required skills, education level, and years of experience from the job description below.
If a field is not mentioned, use "Not specified".

Job Title: {job_title}
Job Description: {job_description}"""),
])

requirements_chain = requirements_template | llm.with_structured_output(JobRequirements)

In [22]:
# Show this works for a sample datapoint
sample_requirements = requirements_chain.invoke({
    "job_title": sample_job_title,
    "job_description": sample_job_desc
})
sample_requirements

JobRequirements(Required_Skills=['Flutter'], Education_Required='Not specified', Experience_Required='1 year (Preferred)')

## Step 4 & 5: Apply the LLM Chain to Each Job Posting and Update the DataFrame

In [23]:
job_results = []
for idx, row in jobs_df.iterrows():
    try:
        category = job_classification_chain.invoke({
            "categories": ", ".join(job_categories),
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        }).strip()

        requirements = requirements_chain.invoke({
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        })

        job_results.append({
            "Predicted_Category": category,
            "Required_Skills": requirements.Required_Skills,
            "Education_Required": requirements.Education_Required,
            "Experience_Required": requirements.Experience_Required,
        })
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        job_results.append({
            "Predicted_Category": "Not specified",
            "Required_Skills": [],
            "Education_Required": "Not specified",
            "Experience_Required": "Not specified",
        })

job_results_df = pd.DataFrame(job_results)
job_results_df.head()

Row 20 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1yjsx9eegkv40b60kgsejcn` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6950, Requested 1126. Please try again in 570ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Row 22 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1yjsx9eegkv40b60kgsejcn` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7200, Requested 1453. Please try again in 4.8975s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Row 23 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1yjsx9eegkv40b60kgsejcn` service ti

,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Technology/IT,"[Python, API development (REST/RPC), Django, F...",Not specified,Not specified
2,Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, R...",Not specified,"5+ years web development, 2+ years recent Reac..."


In [24]:
# Final merged dataframe with all original and new columns together
jobs_final_df = pd.concat([jobs_df.reset_index(drop=True), job_results_df.reset_index(drop=True)], axis=1)
jobs_final_df.head()

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, API development (REST/RPC), Django, F...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, R...",Not specified,"5+ years web development, 2+ years recent Reac..."


In [25]:
jobs_final_df.to_csv("part2_job_analysis_results.csv", index=False)
jobs_final_df.shape

(25, 6)

---
## Notes
- Both parts use `llm.with_structured_output(...)` with a Pydantic schema so the model's output is validated/parsed automatically (same pattern as `3. Structured Output Generation.ipynb`).
- Each row is wrapped in a `try/except` so a single rate-limited/failed call doesn't stop the whole loop - failed rows fall back to `"Not specified"`.
- Bonus (running on the full dataset instead of the first 30/25 rows) was left out here due to time constraints - see the Bonus section below, which attempts it via OpenRouter.

---
# Bonus: Full-Dataset Run via OpenRouter (Llama 3.1 8B)

The Notes above mention the bonus (full dataset instead of the first 30/25 rows) was skipped due to time constraints. This section attempts it using **`meta-llama/llama-3.1-8b-instruct` via [OpenRouter](https://openrouter.ai)** instead of Groq - it reuses the exact same prompts, categories, and Pydantic schemas defined above (`ArticleAnalysis`, `JobRequirements`); only the LLM backend changes. OpenRouter exposes an OpenAI-compatible endpoint, so this uses `langchain_openai.ChatOpenAI` pointed at OpenRouter's `base_url` rather than a new SDK.

**Scale:** ~6,800 total LLM calls across both parts (2,225 news articles + 2,277 job postings x 2 calls each). This runs against a hosted API rather than local hardware, so there's no GPU/CPU decision to make here - but it does mean real network latency, rate limits, and per-token cost. Every loop below still **checkpoints each result to a `.jsonl` file as it completes**, so the run is safely resumable if interrupted - re-running a cell just skips rows already checkpointed.

## Setup
Set an `OPENROUTER_API_KEY` (get one at [openrouter.ai/keys](https://openrouter.ai/keys)) - same pattern as the `GROQ_API_KEY` cell at the top: picked up from `Day2/.env` automatically via `load_dotenv()`, or prompted for below if not set.

In [26]:
if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API Key: ")

In [27]:
from langchain_openai import ChatOpenAI

openrouter_model_name = "meta-llama/llama-3.1-8b-instruct"
# max_tokens caps how many tokens a single call can generate, bounding worst-case
# latency/cost per call - generous headroom over the ~170 tokens these calls need.
openrouter_llm = ChatOpenAI(
    model=openrouter_model_name,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=500,
)

# method="function_calling": the default strict JSON-schema mode
# (llm.with_structured_output(Schema)) causes this model, via OpenRouter, to ramble
# indefinitely instead of terminating - confirmed by testing (hit the max_tokens cap
# verbatim at both 500 and 1500 without producing valid output). Tool-calling mode is
# far more reliable for this model/host combination - verified working at ~0.2-1.3s/call.
#
# Reuse the exact prompt templates and Pydantic schemas from Parts 1 & 2 above -
# only the LLM backend changes.
openrouter_analysis_chain = analysis_template | openrouter_llm.with_structured_output(ArticleAnalysis, method="function_calling")
openrouter_job_classification_chain = job_classification_template | openrouter_llm | StrOutputParser()
openrouter_requirements_chain = requirements_template | openrouter_llm.with_structured_output(JobRequirements, method="function_calling")

In [28]:
# Show this works for a sample datapoint (same pattern as Parts 1 & 2)
sample_openrouter_analysis = openrouter_analysis_chain.invoke({
    "categories": ", ".join(topic_categories),
    "article": sample_article
})
sample_openrouter_analysis

ArticleAnalysis(Detected_Topic='Business', Summary="TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet connections and higher advert sales. The company also benefited from one-off gains, despite a profit dip at Warner Bros and less users for AOL. TimeWarner now owns 8% of Google and is projecting operating earnings growth of around 5% for 2005.", Key_Entities=['TimeWarner', 'Google', 'Warner Bros', 'AOL', 'Bertelsmann', 'Richard Parsons', 'US Securities Exchange Commission (SEC)'])

## Resumable, checkpointed, parallel runner
`max_workers` requests are fired at once with a `ThreadPoolExecutor` instead of looping one row at a time - OpenRouter handles concurrent requests fine, and this meaningfully cuts wall-clock time versus one-at-a-time. A conservative default of 6 is used since the account's exact rate limit isn't known upfront; transient failures (429s, brief network errors) are retried with exponential backoff before a row is given up on.

Each result is still appended to a `.jsonl` checkpoint file the moment it's produced (writes are lock-protected since multiple threads finish concurrently). Re-running a cell reloads whatever's already checkpointed and only processes the remaining rows - so an interrupted run can just be resumed instead of restarted.

In [29]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm.auto import tqdm

def run_with_checkpoint(checkpoint_path, n_rows, call_fn, fallback, max_workers=6, max_retries=3):
    """Invoke call_fn(idx) for idx in range(n_rows) using up to max_workers concurrent
    requests, retrying transient failures (rate limits, network hiccups) up to
    max_retries times with exponential backoff before falling back. Rows already
    present in checkpoint_path (a .jsonl file) are skipped, and each new result is
    appended as soon as it completes, so the run is safely resumable if interrupted."""
    checkpoint_path = Path(checkpoint_path)
    done = {}
    if checkpoint_path.exists():
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    done[rec["idx"]] = rec

    todo = [idx for idx in range(n_rows) if idx not in done]
    write_lock = Lock()

    def worker(idx):
        for attempt in range(max_retries):
            try:
                return idx, {"idx": idx, **call_fn(idx)}
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"Row {idx} failed after {max_retries} attempts: {e}")
                    return idx, {"idx": idx, **fallback}
                time.sleep(2 ** attempt)

    with open(checkpoint_path, "a", encoding="utf-8") as f, \
         ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(worker, idx) for idx in todo]
        for future in tqdm(as_completed(futures), total=len(futures), desc=checkpoint_path.stem):
            idx, record = future.result()
            done[idx] = record
            with write_lock:
                f.write(json.dumps(record) + "\n")
                f.flush()

    return [done[idx] for idx in range(n_rows)]

## Part 1 (bonus): Full news dataset
No `.head(30)` this time - all 2,225 articles.

In [30]:
news_df_full = pd.read_csv("bbc-news-data.csv", sep="\t")
print(news_df_full.shape)

def part1_call(idx):
    row = news_df_full.loc[idx]
    analysis = openrouter_analysis_chain.invoke({
        "categories": ", ".join(topic_categories),
        "article": row["content"],
    })
    return analysis.model_dump()

part1_fallback = {"Detected_Topic": "Not specified", "Summary": "Not specified", "Key_Entities": []}

part1_records = run_with_checkpoint(
    "part1_bonus_checkpoint.jsonl", len(news_df_full), part1_call, part1_fallback
)

(2225, 4)


part1_bonus_checkpoint: 0it [00:00, ?it/s]

In [31]:
news_full_results_df = pd.DataFrame(part1_records).drop(columns="idx")
news_final_full_df = pd.concat(
    [news_df_full.reset_index(drop=True), news_full_results_df.reset_index(drop=True)], axis=1
)
news_final_full_df.to_csv("part1_news_analysis_full_openrouter.csv", index=False)
news_final_full_df.shape

(2225, 7)

## Part 2 (bonus): Full job postings dataset
No `.head(25)` this time - all 2,277 postings (2 calls each: category + requirements).

In [32]:
jobs_df_full = pd.read_csv("job_title_des.csv").rename(
    columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}
)[["Job_Title", "Job_Description"]]
print(jobs_df_full.shape)

def part2_call(idx):
    row = jobs_df_full.loc[idx]
    category = openrouter_job_classification_chain.invoke({
        "categories": ", ".join(job_categories),
        "job_title": row["Job_Title"],
        "job_description": row["Job_Description"],
    }).strip()
    requirements = openrouter_requirements_chain.invoke({
        "job_title": row["Job_Title"],
        "job_description": row["Job_Description"],
    })
    return {
        "Predicted_Category": category,
        "Required_Skills": requirements.Required_Skills,
        "Education_Required": requirements.Education_Required,
        "Experience_Required": requirements.Experience_Required,
    }

part2_fallback = {
    "Predicted_Category": "Not specified",
    "Required_Skills": [],
    "Education_Required": "Not specified",
    "Experience_Required": "Not specified",
}

part2_records = run_with_checkpoint(
    "part2_bonus_checkpoint.jsonl", len(jobs_df_full), part2_call, part2_fallback
)

(2277, 2)


part2_bonus_checkpoint: 0it [00:00, ?it/s]

In [33]:
jobs_full_results_df = pd.DataFrame(part2_records).drop(columns="idx")
jobs_final_full_df = pd.concat(
    [jobs_df_full.reset_index(drop=True), jobs_full_results_df.reset_index(drop=True)], axis=1
)
jobs_final_full_df.to_csv("part2_job_analysis_full_openrouter.csv", index=False)
jobs_final_full_df.shape

(2277, 6)

---
### Bonus notes
- With 6-way concurrent requests and ~6,800 calls total, expect the exact runtime to depend on OpenRouter's rate limits for this key/model - retries with backoff absorb 429s rather than failing rows outright, so a stricter limit shows up as a slower but still-completing run, not errors.
- If Jupyter, the kernel, or the machine gets interrupted partway through, just re-run the `run_with_checkpoint(...)` cell for that part - it resumes from `part1_bonus_checkpoint.jsonl` / `part2_bonus_checkpoint.jsonl` instead of starting over.
- This costs real (small) money per token via OpenRouter, unlike the local-Ollama approach - see the cost estimate discussed alongside this notebook for the ballpark for a run at this scale.
- Quality note: `llama-3.1-8b-instruct` is a reasonably capable general model but still noticeably behind `openai/gpt-oss-120b` (via Groq) at nuanced classification and entity extraction - expect more "Other"/generic results on ambiguous inputs. That trade-off (speed + low cost vs. accuracy) is exactly why the assignment defaults to a hosted frontier-scale model for the graded first-30/25-rows runs above and treats the full-dataset pass as a bonus.